# unnoise on Google Colab

**GitHub does not add an “Open in Colab” button to notebook files.** Use the badge in the repo README, paste this link in your browser, or in Colab: *File → Open notebook → GitHub* → `sivaratrisrinivas/unnoise` → `colab/unnoise.ipynb`.

https://colab.research.google.com/github/sivaratrisrinivas/unnoise/blob/main/colab/unnoise.ipynb

---

1. **Runtime → Change runtime type → GPU**
2. Run the cells in order.
3. The last cell waits until port **8000** is open, then prints a **Colab proxy URL** (and a button to open it). **Do not use `localhost:8000` on your own computer** — that is your laptop, not the machine running `app.py`. The misleading localhost link comes from Colab’s old window helper; use the **proxy link** the cell prints instead. You can still use the commented `serve_kernel_port_as_iframe` line to embed the UI in the notebook.

If `git clone` fails (private repo), zip this project on your machine, **Upload** it in Colab, unzip, and `%cd` into the folder instead of cloning.

In [ ]:
!pip -q install -U pip
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install diffusers transformers safetensors pillow

In [ ]:
# Replace with your repo if different; or upload a zip and skip this cell.
!git clone https://github.com/sivaratrisrinivas/unnoise.git
%cd unnoise

In [ ]:
import html as html_module
import json
import os
import socket
import subprocess
import sys
import time

from google.colab.output import eval_js
from IPython.display import HTML, display

# Must match where `git clone` put the repo (previous cell uses %cd unnoise).
REPO = "/content/unnoise"
PORT = 8000
assert os.path.isfile(os.path.join(REPO, "app.py")), (
    f"Expected {REPO}/app.py — run the clone + %cd cell first, or set REPO to your folder."
)

LOG_PATH = "/tmp/unnoise_app.log"
env = os.environ.copy()
env["HOST"] = "0.0.0.0"
env["PORT"] = str(PORT)
env["DEVICE"] = "cuda"
env["UI_MAX_FRAMES"] = "all"
# WARMUP_MODEL=1 loads HF weights *before* the HTTP server binds — first run can take
# many minutes. 0 = server binds fast; first "Generate" loads weights.
env["WARMUP_MODEL"] = "0"

with open(LOG_PATH, "w", encoding="utf-8") as log:
    proc = subprocess.Popen(
        [sys.executable, "app.py"],
        cwd=REPO,
        env=env,
        stdout=log,
        stderr=subprocess.STDOUT,
    )


def wait_for_port(port: int, timeout_s: float = 120) -> bool:
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            with socket.create_connection(("127.0.0.1", port), timeout=1.0):
                return True
        except OSError:
            time.sleep(1)
            print(".", end="", flush=True)
    return False


print("Starting app.py — logging to", LOG_PATH)
print(f"Waiting for port {PORT} (max ~2 min) …", end="", flush=True)
if not wait_for_port(PORT, timeout_s=120):
    print("\n--- server log (tail) ---")
    print(open(LOG_PATH, "r", encoding="utf-8", errors="replace").read()[-8000:])
    raise RuntimeError(f"Nothing listening on {PORT} — see log above.")

print("\nServer is up on the Colab VM.")

# Real URL to your app (not your laptop’s localhost).
proxy_url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
if not str(proxy_url).startswith("http"):
    raise RuntimeError(f"Unexpected proxyPort return value: {proxy_url!r}")

print(
    "\n*** Do NOT open localhost:8000 on your own PC — that is your computer, not Colab. ***\n"
    "Use this link (or the button below):\n",
    proxy_url,
)

safe_href = html_module.escape(proxy_url, quote=True)
display(
    HTML(
        "<p><strong>Open the UI (new tab):</strong></p>"
        f'<p><a href="{safe_href}" target="_blank" rel="noopener" '
        'style="font-size:1.1em">Open unnoise →</a></p>'
        "<p style=\"color:#666\">If popups are blocked, click the link above.</p>"
    )
)

# Optional: try opening a tab (may be blocked by the browser).
try:
    eval_js(f"window.open({json.dumps(proxy_url)}, '_blank')")
except Exception:
    pass

# Optional: embed inside Colab instead of a separate tab:
# from google.colab import output
# output.serve_kernel_port_as_iframe(PORT, path="/", height=900)